# Week 8 Assessment: Agentic Deal Pricer

This notebook builds a small agentic pipeline that:

1. embeds a small deals dataset into a Chroma vector store,
2. retrieves relevant deals with RAG,
3. uses an LLM to summarize and estimate price,
4. executes a simple multi-step plan with tool calls.


In [1]:
import json
import os
from dataclasses import dataclass

import chromadb
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from dotenv import load_dotenv


In [2]:
load_dotenv(override=True)
API_KEY = os.getenv("OPENAI_API_KEY")
BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openrouter.ai/api/v1")
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

if not API_KEY:
    print("Missing OPENAI_API_KEY in .env")
else:
    print("API key loaded")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)


API key loaded


In [3]:
DEALS = [
    {
        "id": "d1",
        "title": "Dell Latitude 7420 i7 Laptop",
        "category": "Electronics",
        "description": "14-inch business laptop, Intel i7, 16GB RAM, 512GB SSD, 2-year warranty, very good condition",
        "ask": 750,
    },
    {
        "id": "d2",
        "title": "Herman Miller Aeron Chair",
        "category": "Office",
        "description": "Ergonomic office chair, fully adjustable, size B, minor scuffs, no tears, mesh intact",
        "ask": 480,
    },
    {
        "id": "d3",
        "title": "Sony A6400 Camera + 16-50mm Lens",
        "category": "Photography",
        "description": "Mirrorless camera, 24MP, 4K video, includes kit lens and two batteries, shutter count low",
        "ask": 680,
    },
    {
        "id": "d4",
        "title": "Ikea Bekant Standing Desk",
        "category": "Office",
        "description": "Electric standing desk, 160x80 cm, white top, black frame, smooth height adjustment",
        "ask": 250,
    },
    {
        "id": "d5",
        "title": "Apple Watch Series 7",
        "category": "Wearables",
        "description": "45mm GPS model, good battery health, includes charger and extra sport band",
        "ask": 220,
    },
]

def deal_text(d):
    return f"Title: {d['title']}\nCategory: {d['category']}\nDescription: {d['description']}"


In [4]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
texts = [deal_text(d) for d in DEALS]
embeddings = embedder.encode(texts, normalize_embeddings=True)

chroma = chromadb.Client()
collection = chroma.get_or_create_collection("deals")
collection.upsert(
    ids=[d["id"] for d in DEALS],
    documents=texts,
    embeddings=embeddings,
    metadatas=[{"title": d["title"], "category": d["category"], "ask": d["ask"]} for d in DEALS],
)
print("Indexed deals:", len(DEALS))


Indexed deals: 5


In [5]:
# Tooling: search
def search_deals(query, k=3):
    q_emb = embedder.encode([query], normalize_embeddings=True)
    results = collection.query(query_embeddings=q_emb, n_results=k, include=["documents", "metadatas"])
    hits = []
    for doc, meta, did in zip(results["documents"][0], results["metadatas"][0], results["ids"][0]):
        hits.append({"id": did, "doc": doc, "meta": meta})
    return hits


In [6]:
def llm_summarize(text):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Summarize the deal in 2-3 bullet points."},
            {"role": "user", "content": text},
        ],
        max_tokens=150,
    )
    return resp.choices[0].message.content.strip()

def llm_estimate_price(text):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Estimate a fair price in USD. Respond with a single number."},
            {"role": "user", "content": text},
        ],
        max_tokens=20,
    )
    return resp.choices[0].message.content.strip()

def send_notification(deal_id, price):
    return f"Notification queued for {deal_id} at price ${price}."


In [7]:
PLANNER_SYSTEM = "You are a planning agent. Return a JSON list of tool calls."
PLANNER_SCHEMA = {
    "tools": [
        {"tool": "search_deals", "args": {"query": "...", "k": 3}},
        {"tool": "llm_summarize", "args": {"text": "..."}},
        {"tool": "llm_estimate_price", "args": {"text": "..."}},
        {"tool": "send_notification", "args": {"deal_id": "d1", "price": "..."}},
    ]
}

def plan(query):
    prompt = (
        "Create a short plan of tool calls to answer the user. "
        "Use search_deals first, then summarize and estimate price. "
        "Return JSON only.\n\n"
        f"Schema example: {json.dumps(PLANNER_SCHEMA)}\n\n"
        f"User query: {query}"
    )
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": PLANNER_SYSTEM},
            {"role": "user", "content": prompt},
        ],
        max_tokens=300,
    )
    text = resp.choices[0].message.content.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"tools": [{"tool": "search_deals", "args": {"query": query, "k": 3}}]}


In [8]:
# Executor
TOOLS = {
    "search_deals": search_deals,
    "llm_summarize": llm_summarize,
    "llm_estimate_price": llm_estimate_price,
    "send_notification": send_notification,
}

def execute_plan(plan_json):
    outputs = []
    context = {}
    for step in plan_json.get("tools", []):
        tool = step.get("tool")
        args = step.get("args", {})
        if tool not in TOOLS:
            outputs.append({"tool": tool, "error": "unknown tool"})
            continue

        if tool == "search_deals":
            hits = TOOLS[tool](**args)
            context["hits"] = hits
            outputs.append({"tool": tool, "result": hits})
        elif tool in ("llm_summarize", "llm_estimate_price"):
            # Replace placeholder or empty text with actual search results
            if ("text" not in args or not args.get("text") or args["text"] == "<search_deals_results>") and context.get("hits"):
                args = {"text": context["hits"][0]["doc"]}
            result = TOOLS[tool](**args)
            context[tool] = result
            outputs.append({"tool": tool, "result": result})
        else:
            result = TOOLS[tool](**args)
            outputs.append({"tool": tool, "result": result})
    return outputs


In [10]:
query = "Find a good deal for a used business laptop with 16GB RAM and estimate a fair price."
plan_json = plan(query)
print("PLAN:\n", json.dumps(plan_json, indent=2))
results = execute_plan(plan_json)
print("\nRESULTS:\n", json.dumps(results, indent=2))


PLAN:
 {
  "tools": [
    {
      "tool": "search_deals",
      "args": {
        "query": "used business laptop 16GB RAM",
        "k": 3
      }
    },
    {
      "tool": "llm_summarize",
      "args": {
        "text": ""
      }
    },
    {
      "tool": "llm_estimate_price",
      "args": {
        "text": ""
      }
    }
  ]
}

RESULTS:
 [
  {
    "tool": "search_deals",
    "result": [
      {
        "id": "d1",
        "doc": "Title: Dell Latitude 7420 i7 Laptop\nCategory: Electronics\nDescription: 14-inch business laptop, Intel i7, 16GB RAM, 512GB SSD, 2-year warranty, very good condition",
        "meta": {
          "category": "Electronics",
          "ask": 750,
          "title": "Dell Latitude 7420 i7 Laptop"
        }
      },
      {
        "id": "d4",
        "doc": "Title: Ikea Bekant Standing Desk\nCategory: Office\nDescription: Electric standing desk, 160x80 cm, white top, black frame, smooth height adjustment",
        "meta": {
          "category": "Office"